# 02 · Preprocessing and evaluation

## Load data

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.metrics import f1_score, make_scorer, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

root = Path.cwd()
if not (root / "data").is_dir():
    root = root.parent

df = pd.read_csv(root / "data/raw/online_shoppers_intention.csv")

## Remove repeated rows before splitting

Keep one copy of each repeated row so identical records cannot appear in both sets. Without session IDs, some could be separate sessions. The raw CSV stays unchanged.

In [2]:
rows_before = len(df)
df = df.drop_duplicates()
pd.Series({
    "Original rows": rows_before,
    "Repeated rows removed": rows_before - len(df),
    "Rows kept": len(df),
})

Original rows            12330
Repeated rows removed      125
Rows kept                12205
dtype: int64

## Stratified 80/20 train/test split

Stratification keeps a similar share of purchases in both sets. Use `random_state=42` to repeat the same split. Keep the test set aside; preprocessing and model selection will use training data only.

In [3]:
X = df.drop(columns="Revenue")
X["Weekend"] = X["Weekend"].astype(int)
y = df["Revenue"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

assert X_train.index.intersection(X_test.index).empty
assert len(X_train) + len(X_test) == len(df)

pd.DataFrame({
    "sessions": [len(y_train), len(y_test)],
    "purchases": [y_train.sum(), y_test.sum()],
    "purchase_percentage": [y_train.mean() * 100, y_test.mean() * 100],
}, index=["Train", "Test"]).round(2)

,sessions,purchases,purchase_percentage
Train,9764,1526,15.63
Test,2441,382,15.65


## Preprocessing pipeline

Scale numeric features for logistic regression and one-hot encode categories, including numeric category codes. `Weekend` is converted to 0/1 before splitting. There are no missing values, so imputation is not needed.

Keep preprocessing inside the model pipeline so it is fitted on each training fold, not the test set. Unknown categories are ignored when transforming new data.

In [4]:
numeric_features = [
    "Administrative", "Administrative_Duration", "Informational",
    "Informational_Duration", "ProductRelated", "ProductRelated_Duration",
    "BounceRates", "ExitRates", "PageValues", "SpecialDay",
]
categorical_features = [
    "Month", "VisitorType", "OperatingSystems", "Browser", "Region", "TrafficType",
]

preprocessor = ColumnTransformer([
    ("numeric", StandardScaler(), numeric_features),
    ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ("weekend", "passthrough", ["Weekend"]),
])

pipeline = Pipeline([("preprocessing", preprocessor)])

## Training cross-validation

Use five stratified folds from the training set only. Keep the same folds for every model. Each model pipeline fits its preprocessing again within each fold.

In [5]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_folds = list(cv.split(X_train, y_train))

pd.DataFrame([
    {
        "fold": fold,
        "training_sessions": len(train_idx),
        "validation_sessions": len(val_idx),
        "validation_purchase_percentage": y_train.iloc[val_idx].mean() * 100,
    }
    for fold, (train_idx, val_idx) in enumerate(cv_folds, start=1)
]).set_index("fold").round(2)

,training_sessions,validation_sessions,validation_purchase_percentage
fold,,,
1,7811,1953,15.67
2,7811,1953,15.62
3,7811,1953,15.62
4,7811,1953,15.62
5,7812,1952,15.62


## Evaluation metrics

Purchase-class F1 is the main metric, supported by precision, recall and accuracy. Use zero when precision, recall or F1 is undefined, such as when the baseline predicts no purchases.

In [6]:
scoring = {
    "f1": make_scorer(f1_score, pos_label=True, zero_division=0),
    "precision": make_scorer(precision_score, pos_label=True, zero_division=0),
    "recall": make_scorer(recall_score, pos_label=True, zero_division=0),
    "accuracy": "accuracy",
}

pd.DataFrame({
    "metric": list(scoring),
    "purpose": [
        "Balance purchase precision and recall (primary)",
        "Share of predicted purchases that are correct",
        "Share of actual purchases found",
        "Share of all predictions that are correct",
    ],
})

,metric,purpose
0,f1,Balance purchase precision and recall (primary)
1,precision,Share of predicted purchases that are correct
2,recall,Share of actual purchases found
3,accuracy,Share of all predictions that are correct


## References

- [scikit-learn: preprocessing numeric and categorical columns](https://scikit-learn.org/stable/auto_examples/compose/plot_column_transformer_mixed_types.html) — scaling, one-hot encoding and pipelines.
- [scikit-learn: cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html) — train/test splits and training-only evaluation.
- [scikit-learn: StratifiedKFold](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedKFold.html) — folds that keep similar class proportions.
- [scikit-learn: metrics and scoring](https://scikit-learn.org/stable/modules/model_evaluation.html) — accuracy, precision, recall, F1 and scorers.